# DATASET ZU

- O `200-20.zip` que os autores da ResACEUnet (Zu et al. 2024) publicaram no Zenodo 20339874: 200 volumes de treino (0–199) e 20 de validação (200–219), 128³ em `.npy`.
- São os 220 volumes do FaultSeg3D (Wu et al. 2019) do `dataset_wu`, bit a bit e na mesma orientação, só renumerados: o treino deles é o 0–199 do Wu embaralhado e a validação é o 200–219 (ver o `README.md` e o `Compare.ipynb` desta pasta).
- Os arquivos vêm em `(inline, xline, tempo)`, padronizados como os do Wu; o `formatAxis` leva para o `(x, z, y)` do repositório.
- Os 220 volumes entram juntos no `DataBase.csv`, e o `1 - Model` faz o split de sempre (200/10/10).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
import glob, os, shutil, cv2, json, sys, zipfile, hashlib, urllib.request
from tqdm import tqdm
from pathlib import Path

sys.path.append('../..')
from Dataset.index import Normalization, TilesBuilder, getFiles, setFolder, showTile

In [ ]:
BASE_PATH = f'../Dataset/{Path().resolve().name}'
BASE_PATH

In [ ]:
def getNPY(path):
    return np.load(path).astype(np.float32)

OPTIONS = json.loads(open('../../Task/info.json', 'r').read())
OPTIONS['dataset'] = BASE_PATH.split('/')[-1].strip()
OPTIONS

In [ ]:
with open('../../Task/info.json', 'w', encoding='utf-8') as file:
    json.dump(OPTIONS, file, ensure_ascii=False, indent=4) 

# BAIXANDO OS DADOS

- Só baixa quando o `original/` não tem os 220 volumes; confere o md5 publicado no Zenodo e apaga o zip depois de separar.
- Treino e validação dos autores vão juntos para `original/images` e `original/masks` (os `labels` deles), com os nomes deles.

In [ ]:
ZENODO = 'https://zenodo.org/records/20339874/files/200-20.zip?download=1'
MD5    = '9ae85bd4b5846c95e862ec3a200dc37a'

if len(getFiles('original/images')) < 220 or len(getFiles('original/masks')) < 220:
    if not os.path.exists('200-20.zip'):
        urllib.request.urlretrieve(ZENODO, '200-20.zip')

    with open('200-20.zip', 'rb') as file:
        assert hashlib.file_digest(file, 'md5').hexdigest() == MD5, '200-20.zip corrompido: apague e rode de novo'

    setFolder('original/images')
    setFolder('original/masks')

    with zipfile.ZipFile('200-20.zip') as archive:
        for name in [n for n in archive.namelist() if n.endswith('.npy')]:
            folder = 'images' if '/images/' in name else 'masks'
            with archive.open(name) as source, open(os.path.join('original', folder, Path(name).name), 'wb') as target:
                shutil.copyfileobj(source, target)

    os.remove('200-20.zip')

print(len(getFiles('original/images')), 'imagens |', len(getFiles('original/masks')), 'máscaras')

# CONVERTENDO IMAGENS

In [ ]:
images = getFiles('original/images')
masks  = getFiles('original/masks')

print(images[0])
print(masks[0])

In [ ]:
def formatAxis(img):
    return np.transpose(img, (0, 2, 1))

def getImage(base_path, file_name=None):
    path = os.path.join(base_path, file_name) if file_name else base_path
    return formatAxis(getNPY(path))

for i, (img_path, msk_path) in enumerate(zip(images, masks)):
    if i == 6:
        break

    file_name = Path(img_path).name
    print(f'\n\n showing index {i}:', file_name)

    img  = getImage('original/images', file_name)
    mask = getImage('original/masks',  file_name)

    print(img.min(), img.max())
    showTile(img=img)
    showTile(img=img, mask=mask)

### NORMALIZAÇÃO

- `scaling` do `info.json` escolhe o escalonamento da `Normalization` (`Dataset/index.py`); sem a chave é `'normalize'`.
- `'normalize'`: min-max de cada volume para [0, 1].
- `'percentile'`: corte no p01/p99 do conjunto inteiro, reescalado para [0, 1].
- `'standardize'`: padronização de cada volume (média 0, desvio 1), o passo da seção 3.2 do artigo da ResACEUnet.
- `null`: o volume cru, sem escalonamento.
- A escolha vai para a coluna `scaling` do `DataBase.csv`; o `index.py` refaz o `Format` quando a rodada pede outro.

In [ ]:
SCALING   = OPTIONS.get('scaling', Normalization.DEFAULT)
normalize = Normalization(SCALING).fit(images, getImage)

setFolder('images')
setFolder('masks')
print(normalize)

In [ ]:
for img_path, msk_path in tqdm(zip(images, masks), total=len(images)):
    img = getImage('original/images', Path(img_path).name)
    msk = getImage('original/masks',  Path(msk_path).name)

    img = normalize(img)
    file_name = Path(img_path).stem
    
    np.save(os.path.join('images', f'{file_name}.npy'), img)
    np.save(os.path.join('masks',  f'{file_name}.npy'), msk)

In [ ]:
df = [] 

for img_path, mask_path in tqdm(zip(getFiles('images'), getFiles('masks'))):
    img  = np.load(img_path)
    mask = np.load(mask_path)
    data = dict()

    data['id'] = Path(img_path).name
    data['img_min']  = np.min(img)
    data['img_max']  = np.max(img)
    data['img_mean'] = np.mean(img)
    data['img_std']  = np.std(img)

    data['msk_min'] = np.min(mask)
    data['msk_max'] = np.max(mask)

    data['shape'] = img.shape
    data['img_path']  = img_path
    data['mask_path'] = mask_path
    data['scaling'] = SCALING
    df.append(data)


df = pd.DataFrame(df)
print(df.img_path.iloc[2])
df

In [ ]:
if OPTIONS.get('img_size') is None:
    df.to_csv('DataBase.csv', index=None)
    display(df)
    sys.exit("Finalizando o programa: sem tiles nessa rodada")

# TILES

In [ ]:
IMG_SIZE = tuple(OPTIONS['img_size'])
IMG_SIZE

In [ ]:
# os tiles saem de images/ e masks/ (ja normalizados e com os eixos corrigidos), nao dos .dat originais
tileImages = getFiles('images')
tileMasks  = getFiles('masks')

tiles = TilesBuilder(target_size=IMG_SIZE, main_dir='tiles')
for img_path, mask_path in tqdm(zip(tileImages, tileMasks), total=len(tileImages)):
    tiles.update(img_path,  'images/', is_mask=False)
    tiles.update(mask_path, 'masks/',  is_mask=True)

In [ ]:
print(len(getFiles('tiles/images')), 'imagens geradas')

In [ ]:
df = pd.DataFrame()
df['img_path']  = getFiles('tiles/images')
df['mask_path'] = getFiles('tiles/masks')

assert len(df) > 0, 'nenhum tile foi gerado'
assert [Path(p).stem for p in df.img_path] == [Path(p).stem for p in df.mask_path], 'tiles de imagem e mascara desalinhados'

stats = []
for path in tqdm(df.img_path, total=len(df)):
    tile = np.load(path)
    stats.append({
        'shape': tile.shape,
        'min':   np.min(tile),
        'max':   np.max(tile),
        'mean':  np.mean(tile),
        'std':   np.std(tile),
    })

for column in ['shape', 'min', 'max', 'mean', 'std']:
    df[column] = [item[column] for item in stats]

shapes = set(df['shape'])
assert shapes == {IMG_SIZE}, f'tiles com shape inesperado: {shapes} (esperado {IMG_SIZE})'

df['scaling'] = SCALING
print(df.img_path.iloc[0])
df.to_csv('DataBase.csv', index=None)
df